# Databricks Project



In [0]:
# Costumized data frame
data=[(1,'aa',30),(2,'bb',40),(3,'cc',50)]
my_schema="id INT,name STRING,marks INT"
df=spark.createDataFrame(data,schema=my_schema)


In [0]:
df.display()

# Access to read data from Azure Datalake

In [0]:
# With this step the databricks can be allow to read data from the datalake (Azure)
# Copy the above key values in the given places

# Secret dbutils

service_credential = dbutils.secrets.get(
    scope='utkarshlscope',
    key='applicationid-secret'
)




spark.conf.set("fs.azure.account.auth.type.datalakeutkarshl.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.datalakeutkarshl.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.datalakeutkarshl.dfs.core.windows.net", "df4507de-c603-441f-a45d-906b68f4e773")
spark.conf.set("fs.azure.account.oauth2.client.secret.datalakeutkarshl.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.datalakeutkarshl.dfs.core.windows.net", "https://login.microsoftonline.com/037e12b0-3666-42ef-9548-b509e4df25c5/oauth2/token")

### Access Data


### DB Untilities

**dbutils.fs()**

In [0]:
dbutils.fs.ls("abfss://source@datalakeutkarshl.dfs.core.windows.net/")


**dbutils.widgets**

In [0]:
# Dropdown widgets
dbutils.widgets.dropdown(
    "item_type",
    "All",
    [
        "All",
        "Baking Goods",
        "Breads",
        "Breakfast",
        "Canned",
        "Dairy",
        "Frozen Foods",
        "Fruits and Vegetables",
        "Hard Drinks",
        "Health and Hygiene",
        "Household",
        "Meat",
        "Others",
        "Seafood",
        "Snack Foods",
        "Soft Drinks",
        "Starchy Foods"
    ],
    "Select Item Type"
)


In [0]:
# Get the value of the user selected
item_type = dbutils.widgets.get("item_type")
print(item_type)

In [0]:
# Dropdown Outlet Type

dbutils.widgets.dropdown(
    "outlet_type",
    "All",
    [
        "All",
        "Grocery Store",
        "Supermarket Type1",
        "Supermarket Type2",
        "Supermarket Type3"
    ],
    "Select Outlet Type"
)

In [0]:
outlet_type = dbutils.widgets.get("outlet_type")
print(outlet_type)

**dbutils.secrets**


In [0]:
#dbutils.secrets.list(scope='utkarshlscope')

In [0]:
#dbutils.secrets.get(scope ='utkarshlscope', key = 'applicationid-secret')

**Data Reading**


In [0]:
df_sales = spark.read.format('csv')\
              .option('header',True)\
                .option('inferSchema', True)\
                    .load('abfss://source@datalakeutkarshl.dfs.core.windows.net/')
                  
            

In [0]:
df_sales.display()

**Schema and Data Types**

In [0]:
df_sales.printSchema()

**Chesk for Null Values**


In [0]:
from pyspark.sql.functions import col, sum

df_sales.select([
    sum(col(column).isNull().cast("int")).alias(column)
    for column in df_sales.columns
]).show()

**Check for duplicate rows**

In [0]:
total_rows = df_sales.count()

unique_rows = df_sales.dropDuplicates().count()

duplicate_rows = total_rows - unique_rows

print(f"Total rows: {total_rows}")
print(f"Unique rows: {unique_rows}")
print(f"Duplicate rows: {duplicate_rows}")

**Check Unique values in Categorical Columns**

In [0]:
from pyspark.sql.functions import countDistinct

for column in df_sales.columns:
    df_sales.select(
        countDistinct(column).alias(f"Unique values in {column}")
    ).show()

In [0]:
# Data qaulity problem in Item Fat Content

df_sales.select("Item_Fat_Content").distinct().show()

In [0]:
df_sales.select("Item_Type").distinct().show()

In [0]:
df_sales.select("Outlet_Size").distinct().show()

In [0]:
df_sales.select("Outlet_Location_Type").distinct().show()

**Numerical column summaries**


In [0]:
df_sales.describe().show()

**Check the count of the Item Visibility stating 0**

In [0]:
df_sales.filter(
    df_sales.Item_Visibility == 0
).count()

In [0]:
df_sales.filter(
    df_sales.Item_Visibility == 0
).show(10)

Key observation: Data Quality assessment

Item fat conetent is not standardised, the values showing variations like Low Fat, LF, low fat, Regular and reg. There are only two content which are Low Fat and Regular, needs to be standardised. These represent inconsistent labels for the same business categories. This is a confirmed data quality issue.


Item visibility has a minimum value of 0, but there no confirmation whether zero is valid, but this is not an error either. There may be no recorded display on the product. Or there be null instead of 0.The  zero values cannot be confirmed from the available dataset and requires further investigation before any transformation is applied.

Null values/ missing values in Item weight and Outlet size which need to investigate.




# Investigating Data Quality assessment

**1. Missing Values**

In [0]:
# Investigating missing values in Item weight and Outlet Size


from pyspark.sql.functions import col, sum

df_sales.groupBy("Item_Identifier")\
    .agg(
        sum(col("Item_Weight").isNull().cast("int")).alias("Missing_Weight")
    )\
    .orderBy("Missing_Weight", ascending=False)\
    .show()

In [0]:
df_sales.groupBy("Outlet_Identifier")\
    .agg(
        sum(col("Outlet_Size").isNull().cast("int")).alias("Missing_Outlet_Size")
    )\
    .orderBy("Missing_Outlet_Size", ascending=False)\
    .show()

Key observation of missing values:

The item identifier has two rows where Item Weight is null. This tells us that these products have 2 records with missing weight.

In the outlet size the top 3 identifiers have large number of records having null.

In [0]:
# Checking missing values of Item weight having distinct values or not

df_sales.filter(
    col('Item_Identifier') == 'FDA55'
). select ("Item_Identifier",
           "Outlet_Identifier",
           "Item_Weight").show()

In [0]:
df_sales.filter(
    col('Item_Identifier') == 'NCS06'
). select ("Item_Identifier",
           "Outlet_Identifier",
           "Item_Weight").show()

In [0]:
df_sales.filter(
    col('Item_Identifier') == 'FDX20'
). select ("Item_Identifier",
           "Outlet_Identifier",
           "Item_Weight").show()

In [0]:
# Checking only one distinct non-null weight in all items

df_sales.groupBy("Item_Identifier") \
    .agg(
        countDistinct("Item_Weight").alias("Unique_Weights")
    ) \
    .orderBy("Unique_Weights", ascending=False) \
    .show()

Key observation on the Item weight missing values:

All the Item identifier which are having missing valuess haave consistent known weight acrkss it's non-null records.

The missing values can reasonably be poplulated with non-null records.

In [0]:
# Checking whether those outlets have any non-null Outlet Size values

df_sales.groupBy("Outlet_Identifier", "Outlet_Size") \
    .count() \
    .orderBy("Outlet_Identifier") \
    .show()

In [0]:
# Discovering the relation between the Outlet Informations

df_sales.select(
    "Outlet_Identifier",
    "Outlet_Size",
    "Outlet_Location_Type",
    "Outlet_Type"
).distinct() \
 .orderBy("Outlet_Identifier") \
 .show()

In [0]:
df_sales.select(
    "Outlet_Location_Type",
    "Outlet_Size",
    "Outlet_Type"
).distinct() \
 .orderBy("Outlet_Location_Type").show()

Key observation on Outlet size missing values:

Missing outlet size values occur in outlets located in Tier 2 and Tier 3 locations, and these outlets are classified as either Grocery Store or Supermarket Type1.

There no relationship identified that allows the missing outlet sized to determined with confidence, the missing values can be assigned based on the location or outlet type.

**2. Item Visiblity minimum value has 0**

In [0]:
# Checking which item-oulet combinations have zero visibility

df_sales.filter(
    df_sales.Item_Visibility == 0
). select ("Item_Identifier",
           "Outlet_Identifier",
           "Item_Visibility").show()

In [0]:
df_sales.filter(
    col("Item_Identifier") == "FDX20"
).select(
    "Item_Identifier",
    "Outlet_Identifier",
    "Item_Visibility"
).show()

In [0]:
from pyspark.sql.functions import min, max

df_sales.groupBy("Item_Identifier") \
    .agg(
        min("Item_Visibility").alias("Min_Visibility"),
        max("Item_Visibility").alias("Max_Visibility")
    ) \
    .filter(
        (col("Min_Visibility") == 0) &
        (col("Max_Visibility") > 0)
    ) \
    .show()

# Data Cleaning

In [0]:
# Standardising the inconsistent Item Fat Content values

from pyspark.sql.functions import when

df_sales = df_sales.withColumn(
    "Item_Fat_Content",
    when(col("Item_Fat_Content").isin("LF", "low fat"), "Low Fat")
    .when(col("Item_Fat_Content") == "reg", "Regular")
    .otherwise(col("Item_Fat_Content"))
)

In [0]:
df_sales.select("Item_Fat_Content").distinct().show()

In [0]:
# Filling missing values in Item Weight

# Create a lookup table

from pyspark.sql.functions import first

item_weight_lookup = df_sales.groupBy("Item_Identifier") \
    .agg(
        first("Item_Weight", ignorenulls=True).alias("Known_Item_Weight")
    )

In [0]:
# Join the lookup table with the original table

df_sales = df_sales.join(
    item_weight_lookup,
    on="Item_Identifier",
    how="left"
)


In [0]:
# Fill the missing values only

df_sales = df_sales.withColumn(
    "Item_Weight",
    when(
        col("Item_Weight").isNull(),
        col("Known_Item_Weight")
    ).otherwise(col("Item_Weight"))
)

In [0]:
# Removing the temporary lookup column

df_sales = df_sales.drop("Known_Item_Weight")


In [0]:
# Validating the Changes in Item Weight

df_sales.filter(
    col("Item_Weight").isNull()
).count()

In [0]:
df_sales.filter(
    col("Item_Weight").isNull()
).select(
    "Item_Identifier",
    "Outlet_Identifier",
    "Item_Weight"
).show()

In [0]:
df_sales.filter(
    col("Item_Weight").isNull()
).select(
    "Item_Identifier",
    "Item_Type",
    "Item_Fat_Content",
    "Item_MRP",
    "Outlet_Identifier"
).show()

In [0]:
df_sales_clean = df_sales.filter(
    col("Item_Weight").isNotNull()
)

In [0]:
df_sales_clean.filter(
    col("Item_Weight").isNull()
).count()

In [0]:
print("Before:", df_sales.count())
print("After:", df_sales_clean.count())

Key observation:

Four records contained missing Item Weight values for which no corresponding non-null weight was available for the same Item Identifier.

As the missing values could not be reliably recovered from the available data and represented only 0.05% of the dataset, these records were removed to maintain a complete Item Weight field for subsequent analysis.